# Task 1: Data Inspection and Preparation

## a) Understanding and Pruning the Data

loading the data

In [2]:
import pandas as pd

orders = pd.read_csv('orders.csv')
order_products = pd.read_csv('order_products.csv')
aisles = pd.read_csv('aisles.csv')
products = pd.read_csv('products.csv')
departments = pd.read_csv('departments.csv')

checking heads

In [3]:
display(orders.head())
display(order_products.head())
display(aisles.head())
display(products.head())
display(departments.head())

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,NaN
1,2398795,1,2,3,7,15.0
2,473747,1,3,3,12,21.0
3,2254736,1,4,4,7,29.0
4,431534,1,5,4,15,28.0


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


sampling users

In [5]:
SEED = 333
SAMPLE_SIZE = 10000

# original item counts
print("BEFORE FILTERING")
print(f"len(orders): {len(orders)}")
print(f"len(order_products): {len(order_products)}")

# random sampling
sample_user_id = orders['user_id'].drop_duplicates().sample(n=SAMPLE_SIZE, random_state=SEED)

# filter using the sampling
orders_reduced = orders[orders['user_id'].isin(sample_user_id)]
order_products_reduced = order_products[order_products['order_id'].isin(orders_reduced['order_id'])]

# new item counts
print("AFTER FILTERING")
print(f"len(orders): {len(orders_reduced)}")
print(f"len(order_products): {len(order_products_reduced)}")

BEFORE FILTERING
len(orders): 3421083
len(order_products): 33819106
AFTER FILTERING
len(orders): 164035
len(order_products): 1625000


## b) Constructing Transactions

constructing transactions

In [6]:
# merge with products (for product names)
df_merged = order_products_reduced.merge(products[['product_id', 'product_name']], on='product_id')

# group by order number
transactions = df_merged.groupby('order_id')['product_name'].apply(list).values.tolist()

# check head
print("example transaction:")
print(transactions[SEED])

example transaction:
['Smartwater', 'Ginger Beer', 'Pure Irish Butter', 'Dha Omega 3 Vitamin D Milk', 'Selects Smoked Uncured Bacon', 'Green Beans', 'Turkey Bacon', 'Baby Spinach', 'Thick Cut Bacon', 'Red Peppers', 'Hazelnut Spread With Skim Milk & Cocoa', 'Veri Veri Teriyaki Marinade & Sauce', 'Traditional Favorites Tomato & Basil Pasta Sauce', 'Sliced Provolone Cheese', 'Freshly Shaved Parmesan Cheese', 'Organic Yellow Onion']


# Task 2: Mining Association Rules

## a) Exploring the Dataset

generate ruleset

In [7]:
from apyori import apriori

MIN_SUPPORT = 0.001
MIN_CONFIDENCE = 0.2

rules = apriori(transactions, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE, min_lift=3)
results = list(rules)

print(f"len(results): {len(results)}")

len(results): 251


convert ruleset to dataframe

In [17]:
def extract_rules(rules_as_list):
    rules_data = []
    for result in rules_as_list:
        for ordered_stat in result.ordered_statistics:
            rules_data.append({
                'Base': list(ordered_stat.items_base),
                'Add': list(ordered_stat.items_add),
                'Support': result.support,
                'Confidence': ordered_stat.confidence,
                'Lift': ordered_stat.lift
            })
    return pd.DataFrame(rules_data)

rules_df = extract_rules(results)
display(rules_df.sort_values(by='Lift', ascending=False).head(10))

,Base,Add,Support,Confidence,Lift
1,[Almond Milk Strawberry Yogurt],[Almond Milk Blueberry Yogurt],0.001010,0.552901,375.726702
0,[Almond Milk Blueberry Yogurt],[Almond Milk Strawberry Yogurt],0.001010,0.686441,375.726702
6,[Blueberry Yoghurt],[Raspberry Yoghurt],0.001110,0.426859,128.679387
7,[Raspberry Yoghurt],[Blueberry Yoghurt],0.001110,0.334586,128.679387
247,"[Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry...",[Icelandic Style Skyr Blueberry Non-fat Yogurt],0.001079,0.607018,112.284243
244,[Vanilla Skyr Nonfat Yogurt],"[Non Fat Raspberry Yogurt, Icelandic Style Sky...",0.001079,0.204009,103.866708
245,"[Non Fat Raspberry Yogurt, Icelandic Style Sky...",[Vanilla Skyr Nonfat Yogurt],0.001079,0.549206,103.866708
246,"[Vanilla Skyr Nonfat Yogurt, Icelandic Style S...",[Non Fat Raspberry Yogurt],0.001079,0.522659,101.601666
243,[Non Fat Raspberry Yogurt],"[Vanilla Skyr Nonfat Yogurt, Icelandic Style S...",0.001079,0.209697,101.601666
19,[Non Fat Acai & Mixed Berries Yogurt],[Icelandic Style Skyr Blueberry Non-fat Yogurt],0.001035,0.446237,82.543470


## b) Identifying Market Insights

product rule with the highest confidence & support

In [24]:
high_confidence = rules_df[rules_df['Confidence'] > 0.4].sort_values(by='Confidence', ascending=False)
high_support = rules_df.sort_values(by='Support', ascending=False)

print("highest confidence rule:")
display(high_confidence.head(1))
print("highest support rule:")
display(high_support.head(1))

highest confidence rule:


,Base,Add,Support,Confidence,Lift
0,[Almond Milk Blueberry Yogurt],[Almond Milk Strawberry Yogurt],0.00101,0.686441,375.726702


highest support rule:


,Base,Add,Support,Confidence,Lift
27,[Limes],[Large Lemon],0.008836,0.203563,4.158773


top sunday morning rule

In [36]:
# filter for sunday morning
filtered_ids = orders_reduced[
    (orders_reduced['order_dow'] == 0) &
    (orders_reduced['order_hour_of_day'].between(6, 10))
]['order_id']

# transaction list
filtered_df = df_merged[df_merged['order_id'].isin(filtered_ids)]
filtered_transactions = filtered_df.groupby('order_id')['product_name'].apply(list).values.tolist()

# generate ruleset
MIN_SUPPORT = 0.004
MIN_CONFIDENCE = 0.2
filtered_rules_results = list(apriori(filtered_transactions, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE))

# convert to df and display
filtered_rules_df = extract_rules(filtered_rules_results)

print(f"len(filtered_rules_df): {len(filtered_rules_df)}")
print("top sunday morning rule:")
display(filtered_rules_df.sort_values(by='Confidence', ascending=False).head(1))

len(filtered_rules_df): 194
top sunday morning rule:


,Base,Add,Support,Confidence,Lift
36,[Pure Sparkling Water],[Bag of Organic Bananas],0.00463,0.5,3.645518


department rule

In [37]:
# merge with departments (for department names)
df_dept = order_products_reduced.merge(products[['product_id', 'department_id']], on='product_id')
df_dept = df_dept.merge(departments, on='department_id')

# aggregate
grouped = df_dept.groupby('order_id')['department']

# construct transactions (and remove duplicates)
dept_transactions = []
for name, group in grouped:
    unique_departments = list(set(group))
    dept_transactions.append(unique_departments)

# generate ruleset
MIN_SUPPORT = 0.1
MIN_CONFIDENCE = 0.5
dept_results = list(apriori(dept_transactions, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE))

# extract and display ruleset
dept_rules_df = extract_rules(dept_results)

# find top confidence rule
top_rule = dept_rules_df.sort_values(by='Confidence', ascending=False).head(1)
print("top departments rule:")
display(top_rule)

top departments rule:


,Base,Add,Support,Confidence,Lift
88,"[pantry, canned goods]",[produce],0.100346,0.906444,1.215212
